# Convert a TIFF experiment to OME-Zarr v0.5

`convert_tiff_to_omezarr` turns an experiment stored by `TiffWriter`
(one TIFF per frame) into one OME-Zarr store. Sections 1 to 4 use
synthetic data and need no microscope. Section 5 converts a real run
from `tiff_writer.ipynb` on the Ti2-E.

**TIFF layout** (one file per frame, one folder per data type):
```
experiment/
├── raw/            000_00000.tiff  (c, y, x)
├── labels/         000_00000.tiff
├── particles/      000_00000.tiff
├── stim_mask/      000_00000.tiff
├── ref/            000_00004.tiff
└── tracks/         0_latest.parquet
```

**OME-Zarr layout** (one store):
```
output/
├── acquisition.ome.zarr/
│   ├── 0/              raw (t, p, c, y, x)
│   └── labels/
│       ├── labels/0    segmentation masks
│       ├── particles/0 tracked labels
│       └── stim_mask/0 stimulation masks
├── ref/                TIFF fallback
└── tracks/             parquet (copied)
```

## 1. Generate synthetic TIFF data

Skip this section if you convert real data.

In [ ]:
import os
import shutil
from datetime import datetime

import numpy as np
import pandas as pd
import tifffile

work = os.path.abspath(
    os.path.join("data", f"convert-{datetime.now():%Y%m%d-%H%M%S}")
)  # data/ next to this notebook
src_path = os.path.join(work, "sample_tiff_experiment")

N_FOV = 2
N_TIMEPOINTS = 5
N_CHANNELS = 2
IMG_H, IMG_W = 128, 128

for folder in ["raw", "labels", "particles", "stim_mask", "ref", "tracks"]:
    os.makedirs(os.path.join(src_path, folder), exist_ok=True)

rng = np.random.default_rng(42)

for fov in range(N_FOV):
    rows = []
    for t in range(N_TIMEPOINTS):
        fname = f"{fov:03d}_{t:05d}"

        raw = rng.integers(0, 4096, (N_CHANNELS, IMG_H, IMG_W), dtype=np.uint16)
        tifffile.imwrite(os.path.join(src_path, "raw", fname + ".tiff"), raw)

        # A grid of square "cells"
        labels = np.zeros((IMG_H, IMG_W), dtype=np.uint16)
        cell_id = 1
        for yi in range(0, IMG_H, 32):
            for xi in range(0, IMG_W, 32):
                labels[yi : yi + 28, xi : xi + 28] = cell_id
                cell_id += 1
        tifffile.imwrite(os.path.join(src_path, "labels", fname + ".tiff"), labels)

        # Tracked labels: same masks, other ids
        particles = (labels + fov * 100).astype(np.uint16)
        particles[labels == 0] = 0
        tifffile.imwrite(
            os.path.join(src_path, "particles", fname + ".tiff"), particles
        )

        # Stimulation on odd frames
        stim_mask = np.zeros((IMG_H, IMG_W), dtype=np.uint8)
        if t % 2 == 1:
            stim_mask[32:96, 32:96] = 255
        tifffile.imwrite(
            os.path.join(src_path, "stim_mask", fname + ".tiff"), stim_mask
        )

        for lbl in range(1, cell_id):
            rows.append(
                {
                    "timestep": t,
                    "fov": fov,
                    "particle": lbl,
                    "label": lbl,
                    "x": float(rng.uniform(0, IMG_W)),
                    "y": float(rng.uniform(0, IMG_H)),
                }
            )

    # Reference image on the last frame only
    ref_img = rng.integers(0, 4096, (IMG_H, IMG_W), dtype=np.uint16)
    ref_fname = f"{fov:03d}_{N_TIMEPOINTS - 1:05d}"
    tifffile.imwrite(os.path.join(src_path, "ref", ref_fname + ".tiff"), ref_img)

    pd.DataFrame(rows).to_parquet(
        os.path.join(src_path, "tracks", f"{fov}_latest.parquet")
    )

for folder in sorted(os.listdir(src_path)):
    print(f"{folder}/  ({len(os.listdir(os.path.join(src_path, folder)))} files)")

## 2. Convert to OME-Zarr

In [ ]:
from faro.core.conversion import convert_tiff_to_omezarr

dst_path = os.path.join(work, "sample_omezarr_output")

zarr_path = convert_tiff_to_omezarr(
    src_path=src_path,
    dst_path=dst_path,
    channel_names=["iRFP", "RFP"],
    label_chunk_t=1,
    label_shard_t=50,
    overwrite=True,
)

## 3. Inspect the OME-Zarr store

In [ ]:
import zarr

store = zarr.open(zarr_path, mode="r")
print(store.tree())

raw = store["0"]
print(f"Raw array: shape={raw.shape}, dtype={raw.dtype}")
ome = store.attrs.get("ome", {})
axes = ome.get("multiscales", [{}])[0].get("axes", [])
print(f"Axes: {[a['name'] for a in axes]}")
channels = ome.get("omero", {}).get("channels", [])
print(f"Channels: {[ch['label'] for ch in channels]}")

for name in store["labels"].attrs.get("labels", []):
    arr = store[f"labels/{name}/0"]
    print(f"labels/{name}: shape={arr.shape}, dtype={arr.dtype}")

## 4. Verify the data

Every frame of the OME-Zarr store must equal its TIFF file.

In [ ]:
def check_conversion(src_path, zarr_path):
    """Compare every TIFF frame with the OME-Zarr store. Returns True if all match."""
    store = zarr.open(zarr_path, mode="r")
    raw_zarr = store["0"]
    multi_fov = "p" in [a["name"] for a in store.attrs["ome"]["multiscales"][0]["axes"]]
    mismatches = []
    for folder in ["raw", "labels", "particles", "stim_mask"]:
        folder_path = os.path.join(src_path, folder)
        if not os.path.isdir(folder_path):
            continue
        arr = raw_zarr if folder == "raw" else store[f"labels/{folder}/0"]
        for fname in sorted(os.listdir(folder_path)):
            parts = fname.removesuffix(".tiff").split("_")
            fov, t = int(parts[0]), int(parts[-1])
            old = tifffile.imread(os.path.join(folder_path, fname))
            new = np.asarray(arr[t, fov] if multi_fov else arr[t])
            if folder == "raw":
                # imaging channels first; stim readouts follow them
                new = new[: old.shape[0]] if old.ndim == 3 else new[0]
            if not np.array_equal(old, new):
                mismatches.append(f"{folder}/{fname}")
    print(f"{len(mismatches)} mismatches", mismatches[:10])
    return not mismatches


assert check_conversion(src_path, zarr_path)

In [ ]:
for fov in range(N_FOV):
    pq_path = os.path.join(dst_path, "tracks", f"{fov}_latest.parquet")
    df = pd.read_parquet(pq_path)
    print(f"FOV {fov}: {len(df)} rows, {df['particle'].nunique()} particles")

ref_dir = os.path.join(dst_path, "ref")
print("ref/:", sorted(os.listdir(ref_dir)) if os.path.isdir(ref_dir) else "-")

## 5. Convert a TIFF run from the Ti2-E

Run `tiff_writer.ipynb` first and paste the path it prints. That run
has stimulation readouts (`stim/`); `store_stim_images=True` adds them
as an extra raw channel.

In [ ]:
TIFF_RUN = None  # e.g. r"data\tiff-20261116-120000", a run of tiff_writer.ipynb

if TIFF_RUN is None:
    print("Set TIFF_RUN to convert a real run.")
else:
    run_zarr = convert_tiff_to_omezarr(
        src_path=TIFF_RUN,
        dst_path=TIFF_RUN + "_omezarr",
        channel_names=["iRFP", "RFP"],
        store_stim_images=True,
        overwrite=True,
    )
    print(zarr.open(run_zarr, mode="r").tree())
    assert check_conversion(TIFF_RUN, run_zarr)

## 6. Clean up

In [ ]:
shutil.rmtree(work)
print(f"Removed {work}")